# 03_default_prediction_model.ipynb

Train and evaluate Logistic Regression and Random Forest, using imputation inside the pipeline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, average_precision_score
import joblib

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Python":
    PROJECT_ROOT = PROJECT_ROOT.parent
df = pd.read_csv(PROJECT_ROOT / "Data" / "Cleaned" / "credit_risk_clean.csv")
num = ["person_age", "person_income", "person_emp_length", "loan_amnt", "int_rate_filled", "loan_percent_income", "cb_person_cred_hist_length"]
cat = ["person_home_ownership", "loan_intent", "loan_grade", "cb_person_default_on_file"]
X, y = df[num + cat], df["loan_status"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Training rows:", len(X_train), "Test rows:", len(X_test))

In [ ]:
preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat),
])
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=42, class_weight="balanced_subsample", n_jobs=-1),
}
fitted = {}
for name, clf in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", clf)]).fit(X_train, y_train)
    fitted[name] = pipe
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    print(f"\n=== {name} ===")
    print(classification_report(y_test, pred, digits=3))
    print("ROC AUC:", round(roc_auc_score(y_test, proba), 3))
    print("PR AUC:", round(average_precision_score(y_test, proba), 3))
    print(confusion_matrix(y_test, pred))

## Why accuracy is not the right metric

With only 21.87% defaults, an all-approve model may appear highly accurate while missing every default. Recall, precision, ROC AUC, and PR AUC are more informative.

In [ ]:
rf = fitted["Random Forest"]
cat_transformer = rf.named_steps["prep"].named_transformers_["cat"]
names = num + list(cat_transformer.get_feature_names_out(cat))
imp = pd.Series(rf.named_steps["model"].feature_importances_, index=names).sort_values(ascending=False)
print("Feature importance")
print(imp.head(10))
fig, ax = plt.subplots(figsize=(8, 6))
imp.head(10).sort_values().plot(kind="barh", color="#1F3B63", ax=ax)
ax.set_title("Random Forest feature importance")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "Images" / "py-feature-importance.png", dpi=150)
plt.show()

In [ ]:
cv = cross_val_score(fitted["Random Forest"], X, y, cv=5, scoring="roc_auc", n_jobs=-1)
print("CV ROC AUC:", cv.round(3))
print("Mean CV ROC AUC:", round(cv.mean(), 3))
artifact_path = PROJECT_ROOT / "Python" / "credit_risk_model.joblib"
joblib.dump({"model": fitted["Random Forest"], "X_test": X_test, "y_test": y_test, "num": num, "cat": cat}, artifact_path)
print("Saved model artifact:", artifact_path)